# Bulwark: build the training dataset (CPU only)

**This notebook only builds data. No GPU, no training.** It downloads Lc0 games, converts them to positions, has
Stockfish 19 label each position (2,000 nodes), and writes training shards (`train/`, `val/`, `test/`) that the GPU
server trains on later.

**Settings (right-hand panel):** Accelerator **None**, Internet **On**. Add the dataset that holds `bulwark.bundle`
as an input (upload `bulwark.bundle` once as a private dataset).

**Run it:** set `SESSION` below, then **Save Version → Save & Run All (Commit)**. Run 5 copies of this notebook at
the same time with `SESSION = 0, 1, 2, 3, 4` (each takes a different, non-overlapping part of the Lc0 games;
~10M positions each, ~50M in total).

**Checkpointing:** progress is saved after every 50,000 positions, and the run stops starting new work at 11 h so
Kaggle saves the output before its 12 h limit. If the last cell says **NOT DONE**: open the notebook, *Add Input →
Your Work → Notebook Output* of the previous version of this same notebook, and Save & Run All again. It picks up
where it stopped.

**When all 5 say DONE:** download each session's `bulwark_data/s<N>/training/` folder and put the contents of all
five `train/`, `val/`, `test/` folders together into `bulwark/data/training/train|val|test/` on the GPU server.
(File names start with `s<N>-`, so nothing collides.)

In [ ]:
# ---- Parameters -------------------------------------------------------------
SESSION = 0             # 0..4: a different value in each copy of this notebook
NUM_SESSIONS = 5        # how many copies run in total (keep 5 unless you change all of them)
TARGET = 10_000_000     # positions to label in this session (5 sessions x 10M = 50M)
SF_NODES = 2000         # Stockfish nodes per position (keep the same in every session)
DEADLINE_H = 11.0       # stop starting new work after this many hours (Kaggle CPU limit is 12 h)

import os, time, glob, json, subprocess, shutil
START = time.time()
WORK = "/kaggle/working"     # saved when the version finishes
TMP = "/kaggle/tmp"          # scratch: code, tools, downloads (not saved)
os.makedirs(TMP, exist_ok=True)
print("CPUs:", os.cpu_count())

In [ ]:
# ---- Setup: code, converter, Stockfish 19, 3-5 piece tablebases (all in /kaggle/tmp) ----
def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

bundles = glob.glob("/kaggle/input/**/bulwark.bundle", recursive=True)
assert bundles, "add the dataset containing bulwark.bundle as an input"
CODE = f"{TMP}/bulwark"
if not os.path.isdir(CODE):
    sh(f"git clone -q {bundles[0]} {CODE}")
sh(f"cd {CODE} && git log --oneline -1")
sh(f"cd {CODE} && cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j{os.cpu_count()} --target lc0_convert > /dev/null")

SF = f"{TMP}/sf/stockfish/stockfish-linux-x86-64-universal"
if not os.path.exists(SF):
    os.makedirs(f"{TMP}/sf", exist_ok=True)
    sh(f"curl -sfL https://github.com/official-stockfish/Stockfish/releases/download/sf_19/stockfish-linux-x86-64-universal.tar.gz | tar xz -C {TMP}/sf")
sh(f"sha256sum {SF}")

TB = f"{TMP}/syzygy"
if len(glob.glob(f"{TB}/*.rtbw")) < 100:
    os.makedirs(TB, exist_ok=True)
    for kind in ("3-4-5-wdl", "3-4-5-dtz"):
        url = f"https://tablebase.lichess.ovh/tables/standard/{kind}/"
        sh(f"curl -sfL {url} | grep -o 'href=\"[^\"]*rtb[wz]\"' | sed 's/href=\"//;s/\"//' | "
           f"sed 's#^#{url}#' | xargs -P 8 -n 1 curl -sfL -O --output-dir {TB}")
n = len(glob.glob(f"{TB}/*.rtbw")); print("tablebase WDL files:", n); assert n >= 100, "tablebase download failed"

In [ ]:
# ---- Build data (resumes automatically from an earlier version's output added as input) ----
cmd = ["python3", "-u", f"{CODE}/kaggle/datagen.py", "--session", str(SESSION), "--num-sessions", str(NUM_SESSIONS),
       "--target", str(TARGET), "--sf-nodes", str(SF_NODES), "--out", WORK, "--tmp", f"{TMP}/data",
       "--restore-root", "/kaggle/input", "--converter", f"{CODE}/build/lc0_convert", "--stockfish", SF,
       "--syzygy", TB, "--workers", str(os.cpu_count()), "--start", str(START), "--deadline-h", str(DEADLINE_H)]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=CODE)
for line in p.stdout:
    print(line, end="")
assert p.wait() == 0, "datagen failed (see the log above); progress up to the last saved piece is in the output"

In [ ]:
# ---- Report ----
rep = json.load(open(f"{WORK}/bulwark_data/s{SESSION}/report.json"))
print(json.dumps(rep, indent=1))
for split in ("train", "val", "test"):
    files = glob.glob(f"{WORK}/bulwark_data/s{SESSION}/training/{split}/*.bin")
    print(split, len(files), "files", round(sum(os.path.getsize(f) for f in files) / 1e9, 2), "GB")
print("\nDONE" if rep["done"] else "\nNOT DONE: add this version's output as an input and run again (it resumes).")